# 晨星风格箱复现 — 华泰证券研报

**研报**: 《晨星风格箱基于基金持仓数据并根据规模、价值成长特性确定基金风格》(2020-08-21)

**分析对象**: 中欧价值精选混合型证券投资基金 A:021181 / C:021182

**核心算法**:
1. 规模因子: $y = 100 \times [1 + \frac{\ln(Cap) - \ln(MST)}{\ln(LMT) - \ln(MST)}]$
2. 价值-成长因子: $x = 100 \times [1 + \frac{VCG - VT}{GT - VT}]$，其中 $VCG = 成长得分 - 价值得分$
3. 基金风格: 持仓股票得分的加权平均

In [ ]:
import sys
import os

# 添加项目根目录
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('环境加载完成 ✅')

## 1. 导入项目模块

In [ ]:
from source.data_loader import (
    get_all_stock_market_cap,
    get_fund_holdings,
    get_stock_financial_data,
    calculate_market_cap_thresholds,
    calculate_vg_thresholds,
)
from source.factor import (
    calculate_stock_size_score,
    calculate_value_score_from_market,
    calculate_growth_score_from_market,
    calculate_vcg_score,
    calculate_stock_vg_score,
    calculate_fund_size_score,
    calculate_fund_vg_score,
    determine_size_style,
    determine_vg_style,
    analyze_fund_style,
)
from source.plot import (
    plot_morningstar_stylebox,
    plot_nav_curve,
    plot_drawdown,
    plot_stock_distribution,
    plot_style_history,
)

print('项目模块加载完成 ✅')

## 2. 配置参数

In [ ]:
# ===== 基金配置 =====
FUND_CODE = '021181'      # 中欧价值精选混合A
FUND_NAME = '中欧价值精选混合A'
GAMMA = 0.5               # 价值-成长风格判定参数

# ===== 路径配置 =====
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'分析基金: {FUND_NAME} ({FUND_CODE})')
print(f'Gamma参数: {GAMMA}')
print(f'输出目录: {OUTPUT_DIR}')

## 3. 获取全A股票市值数据

In [ ]:
%%time
stock_df = get_all_stock_market_cap()
print(f'全A股票数量: {len(stock_df)}')
print(f'市值范围: {stock_df["market_cap"].min()/1e8:.2f}亿 ~ {stock_df["market_cap"].max()/1e8:.2f}亿')
stock_df.head(10)

## 4. 计算市值门槛值 (MST/LMT)

In [ ]:
cap_thresholds = calculate_market_cap_thresholds(stock_df['market_cap'])

print('='*50)
print('市值门槛值（研报核心参数）:')
print(f'  LMT(大中盘门限): {cap_thresholds["LMT"]/1e8:.2f} 亿')
print(f'  MST(中小盘门限): {cap_thresholds["MST"]/1e8:.2f} 亿')
print('='*50)

# 可视化市值分布
fig, ax = plt.subplots(figsize=(10, 4))
sorted_caps = stock_df['market_cap'].sort_values(ascending=False)
cum_ratio = sorted_caps.cumsum() / sorted_caps.sum()

ax.plot(range(len(cum_ratio)), cum_ratio.values, linewidth=1.5)
ax.axhline(y=0.70, color='red', linestyle='--', label='70%线(LMT)')
ax.axhline(y=0.90, color='orange', linestyle='--', label='90%线(MST)')
ax.set_xlabel('股票序号(按市值降序)')
ax.set_ylabel('累计市值占比')
ax.set_title('全A股市值累计分布')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 获取基金持仓数据

In [ ]:
holdings = get_fund_holdings(FUND_CODE)

print(f'基金 {FUND_CODE} 重仓股数量: {len(holdings)}')
print(f'重仓股合计占比: {holdings["pct"].sum():.2f}%')
print()
holdings

## 6. 合并市值数据 & 获取财务指标

In [ ]:
# 合并市值
stock_cap_map = dict(zip(stock_df['code'], stock_df['market_cap']))
stock_name_map = dict(zip(stock_df['code'], stock_df['name']))

holdings['market_cap'] = holdings['stock_code'].map(stock_cap_map)

# 填充缺失名称
mask_no_name = holdings['stock_name'].isna() | (holdings['stock_name'] == '')
if mask_no_name.any():
    holdings.loc[mask_no_name, 'stock_name'] = holdings.loc[mask_no_name, 'stock_code'].map(stock_name_map)

print(f'市值匹配情况: {holdings["market_cap"].notna().sum()}/{len(holdings)} 只')

# 获取财务指标
stock_codes = holdings['stock_code'].tolist()
try:
    fin_data = get_stock_financial_data(stock_codes)
    print(f'财务指标获取: {len(fin_data)} 只')
except Exception as e:
    print(f'财务指标获取失败: {e}，将使用简化计算')
    fin_data = pd.DataFrame({'code': stock_codes})

holdings

## 7. 计算规模因子（研报公式复现）

In [ ]:
# 计算每只持仓股的规模得分y
for idx, row in holdings.iterrows():
    cap = row.get('market_cap', np.nan)
    if not pd.isna(cap) and cap > 0:
        y = calculate_stock_size_score(cap, cap_thresholds['MST'], cap_thresholds['LMT'])
        holdings.at[idx, 'size_score_y'] = y
        holdings.at[idx, 'size_style'] = determine_size_style(y)
    else:
        holdings.at[idx, 'size_score_y'] = np.nan
        holdings.at[idx, 'size_style'] = '未知'

# 计算基金规模得分Y
fund_Y = calculate_fund_size_score(holdings, cap_thresholds)
fund_size_style = determine_size_style(fund_Y)

print('='*55)
print('规模因子计算结果:')
print(f'  基金规模得分 Y = {fund_Y:.2f}')
print(f'  基金规模风格 → {fund_size_style}')
print('='*55)
print()

# 展示各股规模得分
print(f'{"股票":>10} {"市值(亿)":>10} {"规模Y":>8} {"风格":>6}')
print('-' * 40)
for _, row in holdings.iterrows():
    cap_yi = row.get('market_cap', np.nan)
    if not pd.isna(cap_yi):
        cap_yi = f'{cap_yi/1e8:.2f}'
    else:
        cap_yi = 'N/A'
    print(f'{row.get("stock_name",""):>10} {cap_yi:>10} {row.get("size_score_y",np.nan):>8.1f} {row.get("size_style",""):>6}')

## 8. 计算价值-成长因子（研报公式复现）

In [ ]:
# 计算价值得分和成长得分
if not fin_data.empty and len(fin_data) > 0:
    value_scores = calculate_value_score_from_market(fin_data)
    growth_scores = calculate_growth_score_from_market(fin_data)
    vcg_scores = calculate_vcg_score(value_scores, growth_scores)
    
    fin_data['value_score'] = value_scores.values
    fin_data['growth_score'] = growth_scores.values
    fin_data['vcg_score'] = vcg_scores.values
    
    # 计算VG门限
    vg_thresholds = calculate_vg_thresholds(fin_data)
    print(f'VT(价值-混合门限): {vg_thresholds["VT"]:.4f}')
    print(f'GT(混合-成长门限): {vg_thresholds["GT"]:.4f}')
    
    # 映射到持仓
    vcg_map = dict(zip(fin_data['code'], fin_data['vcg_score']))
    for idx, row in holdings.iterrows():
        vcg = vcg_map.get(row['stock_code'], np.nan)
        holdings.at[idx, 'vcg_score'] = vcg
        if not pd.isna(vcg):
            x = calculate_stock_vg_score(vcg, vg_thresholds['VT'], vg_thresholds['GT'])
            holdings.at[idx, 'vg_score_x'] = x
            holdings.at[idx, 'vg_style'] = determine_vg_style(x, GAMMA)
        else:
            holdings.at[idx, 'vg_score_x'] = np.nan
            holdings.at[idx, 'vg_style'] = '未知'
else:
    vg_thresholds = {'VT': -0.5, 'GT': 0.5}
    holdings['vcg_score'] = np.nan
    holdings['vg_score_x'] = 150.0
    holdings['vg_style'] = '平衡型'

# 计算基金VG得分X
fund_X = calculate_fund_vg_score(holdings, vg_thresholds)
fund_vg_style = determine_vg_style(fund_X, GAMMA)

print()
print('='*55)
print('价值-成长因子计算结果:')
print(f'  基金价成得分 X = {fund_X:.2f}')
print(f'  基金价成风格 → {fund_vg_style}')
print(f'  Gamma参数 = {GAMMA}')
print(f'  价值/平衡边界 = {150*(1-GAMMA/3):.1f}')
print(f'  平衡/成长边界 = {150*(1+GAMMA/3):.1f}')
print('='*55)

## 9. 综合风格判定 & 晨星风格箱

In [ ]:
# 完整风格分析
result = analyze_fund_style(holdings, cap_thresholds, vg_thresholds, GAMMA)

print('╔' + '═'*58 + '╗')
print(f'║  基金: {FUND_NAME} ({FUND_CODE})')
print(f'║  规模得分 Y = {result["fund_size_score_Y"]:.2f} → {result["fund_size_style"]}')
print(f'║  价成得分 X = {result["fund_vg_score_X"]:.2f} → {result["fund_vg_style"]}')
print(f'║  ────────────────────────────────')
print(f'║  ★ 综合风格: {result["fund_style"]} ★')
print('╚' + '═'*58 + '╝')

# 绘制晨星风格箱
fig = plot_morningstar_stylebox(
    result, 
    fund_name=FUND_NAME,
    save_path=os.path.join(OUTPUT_DIR, f'{FUND_CODE}_stylebox.png')
)
plt.show()

## 10. 持仓股风格分布散点图

In [ ]:
# 持仓股在风格空间中的分布
fig = plot_stock_distribution(
    result['stock_details'],
    mst=cap_thresholds['MST'],
    lmt=cap_thresholds['LMT'],
    vt=vg_thresholds['VT'],
    gt=vg_thresholds['GT'],
    fund_name=FUND_NAME,
    save_path=os.path.join(OUTPUT_DIR, f'{FUND_CODE}_stock_distribution.png')
)
plt.show()

## 11. 持仓股风格明细表

In [ ]:
# 完整持仓风格明细
details = result['stock_details'].copy()
if 'market_cap' in details.columns:
    details['市值(亿)'] = details['market_cap'].apply(lambda x: f'{x/1e8:.2f}' if pd.notna(x) else 'N/A')

display_cols = ['stock_code', 'stock_name', '市值(亿)', 'pct', 'size_score_y', 'size_style', 'vg_score_x', 'vg_style']
available_cols = [c for c in display_cols if c in details.columns]
details[available_cols]

## 12. 风格得分统计摘要

In [ ]:
# 持仓股风格分布统计
details = result['stock_details']

if not details.empty:
    print('=== 规模风格分布 ===')
    size_dist = details.groupby('size_style')['pct'].sum()
    for style, pct in size_dist.items():
        print(f'  {style}: {pct:.2f}%')
    
    print()
    print('=== 价值-成长风格分布 ===')
    vg_dist = details.groupby('vg_style')['pct'].sum()
    for style, pct in vg_dist.items():
        print(f'  {style}: {pct:.2f}%')
    
    print()
    print('=== 3×3 风格箱分布 ===')
    details['style_box'] = details['size_style'] + details['vg_style']
    box_dist = details.groupby('style_box')['pct'].sum().sort_values(ascending=False)
    for style, pct in box_dist.items():
        bar = '█' * int(pct / 2)
        print(f'  {style:>8}: {pct:5.2f}% {bar}')

## 13. 尝试获取基金净值进行辅助分析

In [ ]:
# 尝试获取基金净值
nav = None
try:
    import efinance as ef
    nav_df = ef.fund.get_fund_net_value(fund_code=FUND_CODE)
    if nav_df is not None and not nav_df.empty:
        nav = pd.Series(
            pd.to_numeric(nav_df.iloc[:, 1], errors='coerce').values,
            index=pd.to_datetime(nav_df.iloc[:, 0])
        ).dropna()
        nav = nav.sort_index()
        print(f'获取到净值数据: {len(nav)} 条')
        print(f'区间: {nav.index[0].date()} ~ {nav.index[-1].date()}')
except Exception as e:
    print(f'净值获取失败: {e}')

if nav is not None and len(nav) > 0:
    fig = plot_nav_curve(nav, fund_name=FUND_NAME)
    plt.show()
    
    fig = plot_drawdown(nav, fund_name=FUND_NAME)
    plt.show()
    
    # 计算绩效指标
    from source.backtest import calculate_returns, calculate_sharpe_ratio, calculate_max_drawdown
    
    returns = calculate_returns(nav)
    total_ret = (1 + returns).prod() - 1
    ann_ret = (1 + total_ret) ** (252 / len(returns)) - 1
    sharpe = calculate_sharpe_ratio(returns)
    max_dd = calculate_max_drawdown(nav)
    
    print(f'\n绩效指标:')
    print(f'  累计收益: {total_ret*100:.2f}%')
    print(f'  年化收益: {ann_ret*100:.2f}%')
    print(f'  夏普比率: {sharpe:.2f}')
    print(f'  最大回撤: {max_dd*100:.2f}%')
else:
    print('净值数据不可用，跳过绩效分析')

## 14. 保存分析结果

In [ ]:
# 保存持仓风格明细
detail_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_style_details.csv')
if not result['stock_details'].empty:
    result['stock_details'].to_csv(detail_path, index=False, encoding='utf-8-sig')
    print(f'✔ 持仓风格明细: {detail_path}')

# 保存门槛值
threshold_path = os.path.join(OUTPUT_DIR, f'{FUND_CODE}_thresholds.csv')
threshold_df = pd.DataFrame([{
    'fund_code': FUND_CODE,
    'fund_name': FUND_NAME,
    'LMT_yi': cap_thresholds['LMT'] / 1e8,
    'MST_yi': cap_thresholds['MST'] / 1e8,
    'VT': vg_thresholds['VT'],
    'GT': vg_thresholds['GT'],
    'size_score_Y': result['fund_size_score_Y'],
    'size_style': result['fund_size_style'],
    'vg_score_X': result['fund_vg_score_X'],
    'vg_style': result['fund_vg_style'],
    'fund_style': result['fund_style'],
    'gamma': GAMMA,
}])
threshold_df.to_csv(threshold_path, index=False, encoding='utf-8-sig')
print(f'✔ 门槛值与风格结果: {threshold_path}')

print('\n分析完成! 🎉')